In [3]:
# --- 필요한 라이브러리 ---
import os
from pathlib import Path
import cv2
import numpy as np
from tqdm.auto import tqdm

# ----------------- 설정값 -----------------
ROOT_DIR = Path("/home/junseok/Downloads/data/2025-08-24-14-49-29")  # 이미지가 들어있는 최상위 폴더(하위 폴더까지 탐색)
OUTPUT_PATH = Path("/home/junseok/Downloads/data/test1.mp4")        # 출력 mp4 파일 경로
FPS = 30                                                            # 프레임레이트
SORT_BY = "name"   # "name" = 파일명 기준 정렬(기본), "mtime" = 수정시간 기준 정렬
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}     # 허용 확장자
FOURCC = "mp4v"   # OpenCV 내장 MPEG-4. (H.264 원하면 'avc1'이나 'H264'지만 코덱 설치 필요할 수 있음)
# -----------------------------------------

def list_images_recursively(root: Path, exts=IMAGE_EXTS):
    files = []
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            if Path(fn).suffix.lower() in exts:
                files.append(Path(dirpath) / fn)
    return files

def natural_key(p: Path):
    """'img10.jpg'가 'img2.jpg' 뒤로 가도록 자연스런 정렬 키."""
    import re
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split(r'(\d+)', str(p))]

# 1) 이미지 파일 수집
all_imgs = list_images_recursively(ROOT_DIR)
if not all_imgs:
    raise FileNotFoundError(f"이미지를 찾지 못했습니다: {ROOT_DIR}")

# 2) 정렬
if SORT_BY == "mtime":
    all_imgs.sort(key=lambda p: p.stat().st_mtime)   # 수정시간 오래된 순
else:
    all_imgs.sort(key=natural_key)                   # 파일명 자연 정렬

print(f"총 이미지 수: {len(all_imgs)}")

# 3) 타겟 프레임 크기 계산(최대 가로, 최대 세로) -> 레터박스 패딩용
sizes = []
for p in tqdm(all_imgs, desc="사이즈 스캔", leave=False):
    img = cv2.imdecode(np.fromfile(str(p), dtype=np.uint8), cv2.IMREAD_UNCHANGED)
    if img is None:
        print(f"[경고] 읽기 실패: {p}")
        continue
    h, w = img.shape[:2]
    sizes.append((w, h))
if not sizes:
    raise RuntimeError("유효한 이미지를 단 한 장도 읽지 못했습니다.")

target_w = max(w for w, h in sizes)
target_h = max(h for w, h in sizes)
print(f"출력 프레임 크기: {target_w}x{target_h}, FPS={FPS}")

# 4) VideoWriter 준비
fourcc = cv2.VideoWriter_fourcc(*FOURCC)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
vw = cv2.VideoWriter(str(OUTPUT_PATH), fourcc, FPS, (target_w, target_h))

if not vw.isOpened():
    raise RuntimeError("VideoWriter를 열지 못했습니다. 코덱/경로 권한을 확인하세요.")

def letterbox(img, target_size):
    """이미지를 target_size 안에 비율 유지하며 리사이즈 + 중앙 패딩(검정)"""
    tgt_w, tgt_h = target_size
    h, w = img.shape[:2]
    # 스케일 계산 (target 안에 '맞추기')
    scale = min(tgt_w / w, tgt_h / h)
    new_w, new_h = int(round(w * scale)), int(round(h * scale))
    # 리사이즈
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    # 캔버스 생성(BGR 3채널)
    if resized.ndim == 2:
        resized = cv2.cvtColor(resized, cv2.COLOR_GRAY2BGR)
    elif resized.shape[2] == 4:
        # RGBA -> BGR
        resized = cv2.cvtColor(resized, cv2.COLOR_BGRA2BGR)
    canvas = np.zeros((tgt_h, tgt_w, 3), dtype=np.uint8)
    # 중앙 배치
    x0 = (tgt_w - new_w) // 2
    y0 = (tgt_h - new_h) // 2
    canvas[y0:y0 + new_h, x0:x0 + new_w] = resized
    return canvas

# 5) 프레임 작성
written = 0
for p in tqdm(all_imgs, desc="비디오 작성"):
    img = cv2.imdecode(np.fromfile(str(p), dtype=np.uint8), cv2.IMREAD_UNCHANGED)
    if img is None:
        print(f"[경고] 건너뜀(읽기 실패): {p}")
        continue
    frame = letterbox(img, (target_w, target_h))
    vw.write(frame)
    written += 1

vw.release()
print(f"완료! 프레임 {written}장을 '{OUTPUT_PATH}'에 저장했습니다.")



총 이미지 수: 686


사이즈 스캔:   0%|          | 0/686 [00:00<?, ?it/s]

출력 프레임 크기: 640x480, FPS=30


비디오 작성:   0%|          | 0/686 [00:00<?, ?it/s]

완료! 프레임 686장을 '/home/junseok/Downloads/data/test1.mp4'에 저장했습니다.


### UTM 좌표(x,y) -> 경도, 위도 좌표계 변환

In [1]:
# -*- coding: utf-8 -*-
# UTM(E,N[,Z]) 텍스트 -> 위도(lat) 경도(lon) 텍스트 변환

import re, os, sys, subprocess
from pathlib import Path

# pyproj 설치/임포트
try:
    from pyproj import Transformer
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyproj"])
    from pyproj import Transformer

# ===== 경로 설정 =====
base_dir = Path("/home/junseok/Downloads/GPS")
in_file  = base_dir / "gps_waypoints_utm.csv"
out_file = base_dir / "123.txt"

# ===== 입력 읽기 & 숫자 파싱 =====
with open(in_file, "r", encoding="utf-8") as f:
    lines = f.read().splitlines()

def parse_numbers(line: str):
    # 부호/소수 포함 숫자만 추출
    nums = re.findall(r'[-+]?(?:\d+\.\d+|\d+)', line)
    return [float(n) for n in nums]

rows = []
for ln in lines:
    vals = parse_numbers(ln)
    if len(vals) >= 2:
        e, n = vals[0], vals[1]  # 고도(Z)가 있어도 여기서는 무시
        rows.append((e, n))

if not rows:
    raise ValueError("입력 파일에서 좌표를 찾지 못했습니다. 줄마다 'E N [Z]' 형식인지 확인하세요.")

# ===== UTM 존 자동 판정(52N -> 51N) =====
def in_korea(lon, lat):
    return (124 <= lon <= 132) and (33 <= lat <= 39)

def pick_crs(sample_e, sample_n):
    for crs in ("EPSG:32652", "EPSG:32651"):
        tf = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)  # (E,N)->(lon,lat)
        lon, lat = tf.transform(sample_e, sample_n)
        if in_korea(lon, lat):
            return crs
    return "EPSG:32652"  # 기본 가정

chosen_crs = pick_crs(rows[0][0], rows[0][1])

# ===== 전체 변환 =====
tf = Transformer.from_crs(chosen_crs, "EPSG:4326", always_xy=True)
lonlat = [tf.transform(e, n) for (e, n) in rows]  # (lon, lat)

# ===== 저장 (위도 경도 순서, 소수 9자리) =====
with open(out_file, "w", encoding="utf-8") as f:
    for lon, lat in lonlat:
        f.write(f"{lat:.9f} {lon:.9f}\n")

print(f"[완료] 입력  : {in_file}")
print(f"[완료] 출력  : {out_file}")
print(f"[참고] 가정/자동 판정된 입력 좌표계: {chosen_crs}")
print(f"[참고] 총 변환 좌표 수: {len(lonlat)}")

# --- 필요 시 고도까지 포함하려면 위 저장 부분을 아래로 교체하세요 ---
# with open(out_file, "w", encoding="utf-8") as f:
#     for (e, n), (lon, lat) in zip(rows, lonlat):
#         f.write(f"{lat:.9f} {lon:.9f} 0.000\n")  # 고도값이 있으면 여기서 채워 넣기

[완료] 입력  : /home/junseok/Downloads/GPS/gps_waypoints_utm.csv
[완료] 출력  : /home/junseok/Downloads/GPS/123.txt
[참고] 가정/자동 판정된 입력 좌표계: EPSG:32652
[참고] 총 변환 좌표 수: 260


### CSV -> TXT 형식으로 변환

In [3]:
# CSV -> TXT (space-separated) 변환 스크립트
import os
import pandas as pd

in_csv = "/home/junseok/Downloads/GPS/gps_waypoints_utm.csv"
out_txt = os.path.splitext(in_csv)[0] + ".txt"  # 확장자만 .txt로 변경

# CSV 읽기 (쉼표 외 구분자일 수도 있으니 한번 더 시도)
try:
    df = pd.read_csv(in_csv, encoding="utf-8")
except Exception:
    df = pd.read_csv(in_csv, sep=None, engine="python", encoding="utf-8")

# ⚙️ 필요하면 특정 컬럼만 선택해서 내보내기 (예시)
# 원하는 컬럼이 있으면 아래처럼 지정하고, 없으면 전체 컬럼 유지
preferred_cols = []  # 예: ['lat_deg', 'lon_deg'] 또는 ['easting_m','northing_m','height_m']
use_cols = preferred_cols if preferred_cols and set(preferred_cols).issubset(df.columns) else df.columns

# TXT로 저장: 공백 구분, 헤더 없이, 소수 포맷 지정 가능
df.to_csv(
    out_txt,
    columns=use_cols,
    sep=" ",        # 공백 구분
    index=False,    # 인덱스 미포함
    header=False,   # 컬럼명 미포함 (원하면 True)
    float_format="%.9f"  # 실수 자리수 (필요에 맞게 조정)
)

print(f"저장 완료: {out_txt}")


저장 완료: /home/junseok/Downloads/GPS/gps_waypoints_utm.txt


In [6]:
# TXT( E N [Z] ) -> PLY(ASCII) 변환기
# - 입력 줄마다: Easting Northing Height  (숫자 3개; Z가 없으면 0으로 처리)
# - 매우 큰 파일도 동작하도록 "두 번 읽기"로 개수 세고(header) 다시 쓰기

from pathlib import Path

# ===== 경로 설정 =====
IN_TXT  = Path("/home/junseok/Downloads/GPS/test1 (UTM좌표계).txt")   # <-- 필요 시 바꾸세요
OUT_PLY = IN_TXT.with_suffix(".ply")                                   # 같은 폴더에 .ply 저장

def iter_points(txt_path):
    """한 줄씩 읽어 [x, y, z]를 생성. 공백/탭/쉼표 모두 구분자로 취급."""
    with open(txt_path, "r", encoding="utf-8") as f:
        for ln in f:
            s = ln.strip()
            if not s:
                continue
            parts = s.replace(",", " ").split()
            nums = []
            for p in parts:
                try:
                    nums.append(float(p))
                except ValueError:
                    pass
            if len(nums) >= 2:
                x, y = nums[0], nums[1]
                z = nums[2] if len(nums) >= 3 else 0.0
                yield x, y, z

# 1) 먼저 유효 포인트 수 세기 (header에 필요)
count = 0
for _ in iter_points(IN_TXT):
    count += 1

# 2) PLY(ASCII) 쓰기
with open(OUT_PLY, "w", encoding="utf-8") as out:
    out.write("ply\n")
    out.write("format ascii 1.0\n")
    out.write(f"element vertex {count}\n")
    out.write("property float x\n")
    out.write("property float y\n")
    out.write("property float z\n")
    out.write("end_header\n")
    for x, y, z in iter_points(IN_TXT):
        out.write(f"{x:.6f} {y:.6f} {z:.6f}\n")

print(f"[OK] 변환 완료  →  {OUT_PLY}")
print(f"[INFO] 총 포인트 수: {count:,}")

[OK] 변환 완료  →  /home/junseok/Downloads/GPS/test1 (UTM좌표계).ply
[INFO] 총 포인트 수: 260
